# Part 5 of 6: Background, Lighting & Image-Quality Robustness

COMP7015 Group Project – Topic 1: Plant Disease Classification. This notebook investigates robustness beyond the clean test set (Sections 14–20): the same test leaves under changed background, colour, lighting and image quality; the ViT retrained on grayscale and background-removed photos; the same leaf on different backgrounds; where the models look; training with random backgrounds as a countermeasure; a left-out photo series and field photos; and how well the models know when they are wrong.

The notebooks run in order: `01_data` → `02_cnn_resnet` → `03_vit` → `04_test_evaluation` → `05_robustness` → `06_discussion`. The shared code lives in the `plantdisease/` package. Finished training runs and predictions are saved under `OUT_DIR/runs/` and reused, so this notebook can be re-run on its own once the earlier ones have run.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from torchvision.utils import make_grid

from plantdisease.backgrounds import add_background_tests, make_test_variants, stripes
from plantdisease.config import CFG, CLASSES, SEEDS, SHORT_NAMES, out_path
from plantdisease.data import (
    IMAGE_SUFFIXES,
    build_cache,
    make_loader,
    prepare,
    sparse_checkout,
)
from plantdisease.evaluate import (
    compare_seeds,
    comparison_table,
    expected_calibration_error,
    leaf_share_table,
    mean_sd,
    per_class_recall,
    risk_coverage,
    sd,
    seed_scores,
    shift_table,
    stable_errors,
)
from plantdisease.experiments import (
    countermeasure_names,
    input_training_names,
    load_final_models,
)
from plantdisease.plots import finish, grouped_bars
from plantdisease.train import (
    load_history,
    predict_labels,
    predict_probs,
    run_seeds,
    train_dataset,
)

# selected on the validation set (notebooks 02, 03)
BEST_CNN, BEST_RESNET, BEST_VIT = load_final_models()
FINAL_MODELS = [BEST_CNN, BEST_RESNET, BEST_VIT]
data = prepare()  # leaf-group split of notebook 01, rebuilt from the image cache
BASES = (BEST_CNN, BEST_RESNET)  # the models with a countermeasure (Section 18)

## 14. Robustness on the Same Test Leaves

PlantVillage photos are taken under controlled conditions: one leaf, a plain background and even lighting. Real field photos are not. We evaluate every final model and seed on **the same test leaves** under these transformations:

| Test variant | Question |
|---|---|
| **background removed** (PlantVillage `raw/segmented`, leaf on black) | Does the model rely on the leaf or on the background? Every class comes from its own photo series with its own background (notebook 01, Section 4), so a model can learn the background instead of the leaf. |
| **grayscale** | How much does the model rely on colour (yellowing, brown lesions, mould) versus shape and texture? |
| **darker (×0.6)** | Robustness to poor lighting |
| **blur (Gaussian, σ = 2)** | Robustness to out-of-focus photos |
| **noise (σ = 20 grey levels)** | Robustness to sensor noise / low light |

None of these variants was seen during training, so a drop measures generalisation beyond the training conditions. Section 15 asks how much of the drop disappears when the ViT is retrained on grayscale or background-removed photos. Sections 16–18 follow the background question: the same leaf on another background, where the models look, and a countermeasure. Section 19 tests photos that were not taken under the training conditions at all.

Scores are the mean ± standard deviation over the three training seeds; the dots in the charts are the individual seeds.

In [ ]:
# Test leaves under every condition (plantdisease/backgrounds.py: make_test_variants)
tv = make_test_variants(data)
VARIANTS, y_rob = tv.variants, tv.y

fig, axes = plt.subplots(1, len(VARIANTS), figsize=(2.4 * len(VARIANTS), 2.6))
for ax, (variant, imgs) in zip(axes, VARIANTS.items(), strict=True):
    ax.imshow(imgs[0].permute(1, 2, 0))
    ax.set_title(variant, fontsize=9)
    ax.axis("off")
finish()

In [ ]:
# (model, seed, condition) -> probabilities
rob_probs = predict_probs(FINAL_MODELS, VARIANTS)
rob_preds = {k: p.argmax(1) for k, p in rob_probs.items()}
rob_scores = {v: seed_scores(FINAL_MODELS, rob_preds, v, y_rob) for v in VARIANTS}

robustness = pd.DataFrame(
    {v: [mean_sd(rob_scores[v][n], 3) for n in FINAL_MODELS] for v in VARIANTS},
    index=FINAL_MODELS,
)
robustness.to_csv(out_path("robustness.csv"))

fig, ax = plt.subplots(figsize=(13, 4))
grouped_bars(
    ax,
    list(VARIANTS),
    FINAL_MODELS,
    [[rob_scores[v][n].mean() for v in VARIANTS] for n in FINAL_MODELS],
    yerr=[[sd(rob_scores[v][n]) for v in VARIANTS] for n in FINAL_MODELS],
    seeds=[[rob_scores[v][n] for v in VARIANTS] for n in FINAL_MODELS],
    jitter=True,
    alpha=0.8,
)
ax.set_ylabel("test macro-F1 (mean ± sd; dots: seeds)")
ax.set_ylim(0, 1.05)
ax.set_title("Robustness on the same test leaves")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
finish("robustness.png")
robustness

In [ ]:
# Background shortcut: share of predictions per class when the background is removed (seeds pooled)
fig, axes = plt.subplots(1, len(FINAL_MODELS), figsize=(6 * len(FINAL_MODELS), 4.2))
for ax, name in zip(axes, FINAL_MODELS, strict=True):
    preds = np.concatenate([rob_preds[(name, s, "background removed")] for s in SEEDS])
    ConfusionMatrixDisplay.from_predictions(
        np.tile(y_rob, len(SEEDS)),
        preds,
        display_labels=SHORT_NAMES,
        normalize="true",
        values_format=".2f",
        cmap="Reds",
        colorbar=False,
        xticks_rotation=30,
        ax=ax,
    )
    ax.set_title(f"{name}\nbackground removed (row-normalised)")
finish("background_removed_confusion.png")

## 15. Retraining on Grayscale and Background-Removed Photos (ViT)

A drop under a test variant mixes two effects: information that is really lost (colour, background) and a **mismatch between training and test input**. Following Zhouyang's input ablation, the selected ViT configuration is retrained twice more with all seeds: once on grayscale photos and once on background-removed photos. Each version is validated (and its checkpoint selected) on the same input version, then every model is tested on all three versions of the same test leaves. The diagonal of the resulting 3×3 table is "trained and tested on the same input"; the off-diagonal cells are input mismatches.

The grayscale photos are computed from the colour photos (three identical channels); the background-removed photos are PlantVillage's own segmentations, which also cut away some dark leaf regions and the thin leaf edge. A loss on the diagonal therefore measures the value of colour, or of the background together with segmentation artefacts.

In [ ]:
INPUT_MODELS = [BEST_VIT, *input_training_names(BEST_VIT)]
for name in INPUT_MODELS[1:]:
    run_seeds(name, data)

TEST_INPUTS = ["original", "grayscale", "background removed"]
input_preds = predict_labels(INPUT_MODELS, {v: VARIANTS[v] for v in TEST_INPUTS})
input_scores = {
    v: seed_scores(INPUT_MODELS, input_preds, v, y_rob) for v in TEST_INPUTS
}
training_input = dict(
    zip(INPUT_MODELS, ["color", "grayscale", "background removed"], strict=True)
)

input_grid = pd.DataFrame(
    {
        f"test: {v}": [mean_sd(input_scores[v][n], 4) for n in INPUT_MODELS]
        for v in TEST_INPUTS
    },
    index=[f"train: {training_input[n]}" for n in INPUT_MODELS],
)
input_grid.to_csv(out_path("input_training_grid.csv"))

grid = np.array(
    [[input_scores[v][n].mean() for v in TEST_INPUTS] for n in INPUT_MODELS]
)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(grid, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(
            j,
            i,
            f"{grid[i, j]:.3f}",
            ha="center",
            va="center",
            color="white" if grid[i, j] > 0.6 else "black",
        )
ax.set_xticks(range(3), TEST_INPUTS)
ax.set_yticks(range(3), list(training_input.values()))
ax.set_xlabel("test input")
ax.set_ylabel("training input")
ax.set_title(f"{BEST_VIT}: test macro-F1 (mean over seeds)", fontsize=9)
finish("input_training_grid.png")
input_grid

In [ ]:
# Matched training and test input: per-class F1 (mean over seeds) and errors that every seed makes
rows = {}
for name, test_input in zip(INPUT_MODELS, TEST_INPUTS, strict=True):
    preds = [input_preds[(name, s, test_input)] for s in SEEDS]
    per_class = np.mean([f1_score(y_rob, p, average=None) for p in preds], axis=0)
    rows[f"{training_input[name]} → {test_input}"] = {
        **dict(zip([f"F1 {s}" for s in SHORT_NAMES], per_class, strict=True)),
        **stable_errors(preds, y_rob),
    }
pd.DataFrame(rows).T

## 16. Is It the Background? The Same Leaf on Two Backgrounds

Removing the background changes more than the background: black pixels and a sharp mask edge appear, and the segmentation sometimes cuts away dark leaf regions or the thin edge of the leaf. Here **only the background changes**. We take the leaf of every test photo (the **leaf mask** is every pixel that is not black in the segmented photo, small gaps closed, holes filled) and place it on a background tile:

| Test variant | Background | Question |
|---|---|---|
| **same-class background** | corner of another test photo of the **same class** | Control: the same cut-out leaf with a background that fits its label. |
| **other-class background** | corner of a test photo of a **different class** | Do the predictions follow the leaf or the background? |

Both versions contain exactly the same leaf pixels, so a difference between them can only come from the background. A model that learned the background predicts the class of the background photo in the second version. The tile is a 32 × 32 corner of another test photo without leaf pixels, mirrored and repeated over the image. We report macro-F1 and the **share of predictions that equal the class of the background photo**. Leaf masks, corner tiles and the composed images are built by `add_background_tests` in `plantdisease/backgrounds.py`.

In [ ]:
# adds tv.masks, tv.keep, tv.background_tests and tv.donor_class
add_background_tests(tv)
BACKGROUND_TESTS, DONOR_CLASS = tv.background_tests, tv.donor_class

# Examples: original photo, same leaf on a background of its own class, on a background of another class
fig, axes = plt.subplots(3, 6, figsize=(13, 6.8))
for col, i in enumerate(np.linspace(0, len(y_rob) - 1, 6).astype(int)):
    for row, (title, imgs) in enumerate(
        {"original": tv.original, **BACKGROUND_TESTS}.items()
    ):
        axes[row, col].imshow(imgs[i].permute(1, 2, 0))
        axes[row, col].set_title(f"{SHORT_NAMES[y_rob[i]]}\n{title}", fontsize=8)
        axes[row, col].axis("off")
finish()

In [ ]:
bg_preds = predict_labels(FINAL_MODELS, BACKGROUND_TESTS)
bg_scores = {v: seed_scores(FINAL_MODELS, bg_preds, v, y_rob) for v in BACKGROUND_TESTS}
rows = []
for name in FINAL_MODELS:
    for variant in BACKGROUND_TESTS:
        follows = [
            (bg_preds[(name, s, variant)] == DONOR_CLASS[variant]).mean() for s in SEEDS
        ]
        rows.append(
            {
                "model": name,
                "variant": variant,
                "macro-F1": mean_sd(bg_scores[variant][name], 3),
                "prediction = class of the background photo": (
                    mean_sd(follows, 3) if variant == "other-class background" else ""
                ),
            }
        )
background_tests = pd.DataFrame(rows)
background_tests.to_csv(out_path("background_swap_tests.csv"), index=False)

scores_of = {"original": rob_scores["original"], **bg_scores}
conditions = list(scores_of)
fig, ax = plt.subplots(figsize=(8, 3.8))
grouped_bars(
    ax,
    FINAL_MODELS,
    conditions,
    [[scores_of[c][n].mean() for n in FINAL_MODELS] for c in conditions],
    seeds=[[scores_of[c][n] for n in FINAL_MODELS] for c in conditions],
)
ax.tick_params(axis="x", labelsize=8)
ax.set_ylabel("test macro-F1 (bars: mean, dots: seeds)")
ax.set_ylim(0, 1.05)
ax.set_title("The same leaves on a background of their own class and of another class")
ax.legend(loc="lower left", fontsize=8)
ax.grid(axis="y", alpha=0.3)
finish("background_swap_tests.png")
background_tests

In [ ]:
# Same leaf, only the background differs: other-class minus same-class background (same seed)
rows = []
donor = DONOR_CLASS["other-class background"]
for name in FINAL_MODELS:
    row = {
        "model": name,
        **compare_seeds(
            bg_scores["other-class background"][name],
            bg_scores["same-class background"][name],
        ),
    }
    for c, short in enumerate(SHORT_NAMES):  # which background pulls the prediction?
        follows = [
            (bg_preds[(name, s, "other-class background")][donor == c] == c).mean()
            for s in SEEDS
        ]
        row[f"follows a {short} background"] = np.mean(follows)
    rows.append(row)
same_vs_other = pd.DataFrame(rows).set_index("model")
same_vs_other.to_csv(out_path("background_swap_paired.csv"))
same_vs_other.round(3)

## 17. Where Do the Models Look? Attention on the Leaf

Notebook 04 showed Grad-CAM maps for six example photos. Here we turn them into one number per model. From the **leaf mask** of Section 16 we compute the **leaf share**: the part of the Grad-CAM weight that lies on the leaf. A map that ignored the leaf would give the leaf's share of the image area (*leaf area share*), so a model that really looks at the leaf has a leaf share clearly above that value. We compute it for the original photos, the photos with the background removed and the same leaves on the background of another class. Grad-CAM needs convolutional feature maps, so the ViT is not included.

In [ ]:
leaf_attention_table = leaf_share_table([BEST_CNN, BEST_RESNET], tv.all_tests, tv.masks)
leaf_attention_table.to_csv(out_path("leaf_attention.csv"))

fig, ax = plt.subplots(figsize=(8, 3.8))
positions = np.arange(len(leaf_attention_table))
ax.bar(
    positions,
    leaf_attention_table["leaf share"],
    yerr=leaf_attention_table["sd"],
    capsize=3,
    label="leaf share of the attention (mean ± sd over seeds)",
)
ax.scatter(
    positions,
    leaf_attention_table["leaf area share"],
    marker="_",
    s=900,
    color="red",
    label="leaf area share (uniform map)",
)
ax.set_xticks(
    positions, [f"{n}\n{c}" for n, c in leaf_attention_table.index], fontsize=8
)
ax.set_ylim(0, 1)
ax.set_ylabel("share of the Grad-CAM weight on the leaf")
ax.set_title("Where do the models look?")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
finish("leaf_attention.png")
leaf_attention_table.round(3)

## 18. Countermeasure: Training with Random Backgrounds

Every class comes from its own photo series, so in the training data the background carries information about the label. As a countermeasure, **the background behind the leaf is replaced by a random one during training**:

| Model | Training | Purpose |
|---|---|---|
| **baseline** | unchanged | reference |
| **countermeasure** | with probability 0.5 everything outside the leaf mask becomes black, a solid colour, a gradient, smooth noise or a leaf-free corner tile of another training photo | the background no longer tells the class |
| **control** | the same cut-out leaves, but the background tile always comes from a photo of the same class | separates label-free backgrounds from seeing cut-out leaves |

The leaf masks come from the segmented photos; photos with an empty mask stay unchanged. The swap is an additional augmentation before the usual ones. The selected CNN and ResNet18 keep all hyper-parameters, the split, the seeds and the checkpoint selection (clean validation macro-F1). The tests are those of Section 16, background removed and diagonal stripes, a background type that never occurs in training. Black is a training background, so "background removed" is not unseen for the countermeasure models.

### 18.1 Leaf masks and the background augmentation

A configuration can carry two optional settings, `background_swap` (the swap probability) and `class_matched_background`. With them, `train_dataset` (`plantdisease/train.py`) uses `BackgroundSwapDataset` (`plantdisease/backgrounds.py`); without them nothing changes.

In [ ]:
# leaf masks of all photos, aligned with data.images
LEAF_MASKS, SWAP_OK = data.swap_masks

# Visualise training images with random backgrounds
swap = {"augment": True, "background_swap": CFG["countermeasure"]["swap_prob"]}
x, y = next(iter(make_loader(train_dataset(swap, data), 16, seed=0)))
plt.figure(figsize=(12, 3.5))
plt.imshow(make_grid(x, nrow=8, padding=2, normalize=True).permute(1, 2, 0))
plt.axis("off")
plt.title("Random backgrounds: " + ", ".join(SHORT_NAMES[c] for c in y[:8]) + ", ...")
plt.show()

### 18.2 Training the countermeasure models

Each selected model is trained twice more with all training seeds: with **random backgrounds** (the countermeasure) and with **class-matched backgrounds** (the control).

In [ ]:
# '<model> + random backgrounds' and '<model> + class-matched backgrounds' (plantdisease/experiments.py: config_of)
countermeasure_models = [name for base in BASES for name in countermeasure_names(base)]
for name in countermeasure_models:
    run_seeds(name, data)

In [ ]:
# Validation curves of the baselines, countermeasures and controls (thin: seeds, thick: mean)
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex="col")
for col, base in enumerate(BASES):
    for k, name in enumerate([base, *countermeasure_names(base)]):
        label = "baseline" if name == base else name[len(base) :].lstrip(" +")
        histories_of = [load_history(name, seed) for seed in SEEDS]
        for row, key in enumerate(("val_loss", "val_f1")):
            values = np.array([h[key] for h in histories_of])
            epochs = np.arange(1, values.shape[1] + 1)
            axes[row, col].plot(epochs, values.T, color=f"C{k}", alpha=0.25)
            axes[row, col].plot(epochs, values.mean(0), color=f"C{k}", label=label)
    axes[0, col].set_title(base, fontsize=9)
    axes[1, col].set_xlabel("epoch")
axes[0, 0].set_ylabel("validation loss")
axes[1, 0].set_ylabel("validation macro-F1")
for ax in axes.ravel():
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
finish("countermeasure_curves.png")

### 18.3 Does the countermeasure help? The same leaves on different backgrounds

Every model is tested on the same test leaves in five conditions:

| Condition | What changes | Question |
|---|---|---|
| **original** | nothing | Does the countermeasure cost accuracy on clean photos? |
| **new background (stripes)** | the background, a type never used in training | Does the model cope with an unseen background? |
| **same-class / other-class background** | only the background (Section 16) | Do the predictions follow the leaf and no longer the background? |
| **background removed** | black background | Secondary check, because black is a training background. |

Each architecture appears as **baseline**, **countermeasure** (random backgrounds) and **control** (class-matched backgrounds). The table gives the macro-F1 (mean ± sd over the seeds). Countermeasure and control use the same seeds as their baseline, so the second table compares them seed by seed: the mean and sd of the difference and in how many seeds the new model is higher.

In [ ]:
new_background = torch.where(
    tv.keep[:, None], tv.original, stripes(len(tv.original), *tv.original.shape[-2:])
)  # the leaf (or the whole photo if its mask is empty) on stripes
EVAL_VARIANTS = {
    "original": VARIANTS["original"],
    "new background (stripes)": new_background,
    **BACKGROUND_TESTS,
    "background removed": VARIANTS["background removed"],
}

compared = [name for base in BASES for name in (base, *countermeasure_names(base))]
# (model, seed, condition) -> predicted classes
variant_preds = predict_labels(compared, EVAL_VARIANTS)
seed_f1 = {v: seed_scores(compared, variant_preds, v, y_rob) for v in EVAL_VARIANTS}

countermeasure_table = pd.DataFrame(
    {v: [mean_sd(seed_f1[v][n], 3) for n in compared] for v in EVAL_VARIANTS},
    index=compared,
)
countermeasure_table.to_csv(out_path("countermeasure_results.csv"))

fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for ax, base in zip(axes, BASES, strict=True):
    group = [base, *countermeasure_names(base)]
    grouped_bars(
        ax,
        list(EVAL_VARIANTS),
        ["baseline", *[n[len(base) :].lstrip(" +") for n in group[1:]]],
        [[seed_f1[v][n].mean() for v in EVAL_VARIANTS] for n in group],
        yerr=[[sd(seed_f1[v][n]) for v in EVAL_VARIANTS] for n in group],
        seeds=[[seed_f1[v][n] for v in EVAL_VARIANTS] for n in group],
    )
    ax.tick_params(axis="x", labelrotation=20, labelsize=8)
    ax.set_title(base)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="lower left", fontsize=8)
axes[0].set_ylabel("test macro-F1 (mean ± sd; dots: seeds)")
finish("countermeasure_results.png")
countermeasure_table

In [ ]:
# Countermeasure / control minus baseline, seed by seed, in every condition
pairs = [(name, base) for base in BASES for name in countermeasure_names(base)]
countermeasure_comparison = pd.concat(
    {v: comparison_table(pairs, seed_f1[v]) for v in EVAL_VARIANTS}, names=["condition"]
).droplevel(1)
countermeasure_comparison.to_csv(out_path("countermeasure_comparison.csv"))
countermeasure_comparison.round(4)

In [ ]:
# Do the predictions follow the leaf or the background? (other-class background, mean over seeds)
condition = "other-class background"
following = pd.DataFrame(
    {
        column: [
            mean_sd(
                [(variant_preds[(n, s, condition)] == target).mean() for s in SEEDS], 3
            )
            for n in compared
        ]
        for column, target in {
            "prediction = class of the leaf": y_rob,
            "prediction = class of the background photo": DONOR_CLASS[condition],
        }.items()
    },
    index=compared,
)
following.to_csv(out_path("countermeasure_background_following.csv"))
following

### 18.4 Does the countermeasure move the attention to the leaf?

The leaf share of Section 17 for the countermeasure models, next to the baselines. If random backgrounds work as intended, the leaf share of the countermeasure models should rise, in particular on backgrounds of another class.

In [ ]:
countermeasure_attention = leaf_share_table(
    countermeasure_models, tv.all_tests, tv.masks
)
leaf_attention_all = pd.concat([leaf_attention_table, countermeasure_attention])
leaf_attention_all.to_csv(out_path("leaf_attention_countermeasure.csv"))
leaf_attention_all.round(3)

## 19. Beyond Controlled Photos

Sections 14–18 change the same test leaves. Here the photo changes as a whole: background, light, camera, the individual plants and the disease stage differ together. We evaluate all trained models without retraining: the final models and the countermeasure models of Section 18. A drop shows that a model does not transfer, not why.

### 19.1 A photo series that was left out of training

The healthy greenhouse series `GH_HL` (notebook 01, Section 3) has no official leaf ids and was never used for training, validation or the clean test. It is the only left-out series of our four classes: early blight, leaf mold and spider mites each have a single series. The test therefore covers **one class**: we report the **recall** (share of `GH_HL` photos predicted healthy) next to the recall of healthy on the clean test, and which class the missed photos go to.

In [ ]:
other_df = data.other_df
SERIES_IMAGES = build_cache(other_df["path"], "other_series")
y_series = other_df["label"].to_numpy()
print(
    f"{len(other_df)} photos of the left-out series: "
    + ", ".join(f"{s} ({n})" for s, n in other_df.series.value_counts().items())
)

shift_models = list(dict.fromkeys([*FINAL_MODELS, *compared]))
TEST_IMAGES = data.images[data.test_idx]
series_preds = predict_labels(
    shift_models, {"test": TEST_IMAGES, "series": SERIES_IMAGES}
)

healthy = CLASSES.index("Tomato___healthy")
rows, series_recall = [], {}
for name in shift_models:
    clean = [
        (series_preds[(name, s, "test")][data.y_test == healthy] == healthy).mean()
        for s in SEEDS
    ]
    series_recall[name] = np.array(
        [(series_preds[(name, s, "series")] == healthy).mean() for s in SEEDS]
    )
    pooled = np.concatenate([series_preds[(name, s, "series")] for s in SEEDS])
    rows.append(
        {
            "model": name,
            "clean test: healthy recall": mean_sd(clean, 3),
            "GH_HL: healthy recall": mean_sd(series_recall[name], 3),
            **{
                f"GH_HL predicted {short}": (pooled == c).mean()
                for c, short in enumerate(SHORT_NAMES)
                if c != healthy
            },
        }
    )
series_results = pd.DataFrame(rows).set_index("model")
series_results.to_csv(out_path("series_shift.csv"))
display(series_results.round(3))

pairs = [(name, base) for base in BASES for name in countermeasure_names(base)]
comparison_table(pairs, series_recall)

### 19.2 Field photos (PlantDoc)

PlantDoc (Singh et al., 2020) collects photos of plants outside the lab: internet photos with several leaves, hands, soil and varying light. We use the four folders that match our classes (`Tomato leaf` is healthy, `Tomato mold leaf` is leaf mold) and **all** their photos as an external test, without retraining. The labels come from web sources and were only checked roughly, so the scores are indicative. **PlantDoc has only 2 spider-mite photos**, so the spider-mite scores are anecdotal; the macro-F1 is computed over the classes that occur, and the per-class recall shows each class separately.

In [ ]:
FIELD, FIELD_ROOT = CFG["field_photos"], CFG["field_root"]
field_folders = [
    f"{s}/{n}" for s in ("train", "test") for n in FIELD["folders"].values()
]
available = sparse_checkout(FIELD["url"], FIELD_ROOT, field_folders)
FIELD_AVAILABLE = len(available) > 0
print("PlantDoc folders available:", available)

In [ ]:
def shift_plot(clean, shifted, *, label, ylabel, title, file_name):
    # Clean test vs. shifted photos per model (bars: mean, dots: seeds of the shifted photos).

    _, ax = plt.subplots(figsize=(11, 3.8))
    names = list(shifted)
    grouped_bars(
        ax,
        [n.replace(" + ", "\n+ ") for n in names],
        ["clean test", label],
        [[clean[n].mean() for n in names], [shifted[n].mean() for n in names]],
        seeds=[None, [shifted[n] for n in names]],
    )
    ax.tick_params(axis="x", labelsize=7)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    finish(file_name)


if FIELD_AVAILABLE:
    field_records = []
    for label, class_name in enumerate(CLASSES):
        # the authors' train/test split is not used: all photos are test data
        for split in ("train", "test"):
            folder = os.path.join(FIELD_ROOT, split, FIELD["folders"][class_name])
            if not os.path.isdir(folder):
                continue
            field_records += [
                {
                    "path": os.path.join(folder, f),
                    "label": label,
                    "class_name": class_name,
                }
                for f in sorted(os.listdir(folder))
                if f.lower().endswith(IMAGE_SUFFIXES)
            ]
    field_df = pd.DataFrame(field_records)
    FIELD_IMAGES = build_cache(field_df["path"], "field")
    y_field = field_df["label"].to_numpy()
    field_present = np.unique(y_field)
    print(field_df["class_name"].value_counts().reindex(CLASSES).to_dict())

    field_preds = predict_labels(
        shift_models, {"test": TEST_IMAGES, "field": FIELD_IMAGES}
    )
    field_results, field_clean, field_seed = shift_table(
        shift_models,
        field_preds,
        "field",
        y=y_field,
        y_clean=data.y_test,
        columns=("clean test", "field photos"),
    )
    field_results.to_csv(out_path("field_photos.csv"))
    shift_plot(
        field_clean,
        field_seed,
        label="field photos",
        ylabel="macro-F1 over the classes present",
        title="Clean test and field photos (bars: mean, dots: seeds)",
        file_name="field_photos.png",
    )
    display(field_results)
else:
    print("Section 19.2 is skipped: the PlantDoc photos are not available.")

In [ ]:
if FIELD_AVAILABLE:
    # Recall per class on the field photos (mean over seeds) and countermeasure vs. baseline

    field_recall = pd.DataFrame.from_dict(
        per_class_recall(shift_models, field_preds, "field", y_field, field_present),
        orient="index",
        columns=[
            f"{SHORT_NAMES[c]} ({(y_field == c).sum()} photos)" for c in field_present
        ],
    )
    field_recall.to_csv(out_path("field_photos_recall.csv"))
    display(field_recall.round(3))
    display(comparison_table(pairs, field_seed).round(4))
else:
    print("Section 19.2 is skipped: the PlantDoc photos are not available.")

## 20. Can the Models Tell When They Are Wrong? Confidence and Abstention

A model that is unsure could hand a photo to an expert instead of guessing. This works only if wrong predictions have a lower confidence than correct ones. We take the highest predicted probability as the **confidence**, sort the test photos by it and let the model **abstain** on the least confident photos. The **risk-coverage curve** shows the accuracy of the photos that are still answered (y axis) against the share of photos that are answered (x axis). A curve that rises quickly when a few photos are left out means that the confidence flags the errors. A flat curve means that the model is as confident when it is wrong as when it is right. We show the curves for the original test images and under the changes of Section 14.

The curves are descriptive: no abstention threshold is chosen on the test set. The table also reports the **expected calibration error** (ECE), the average gap between confidence and accuracy over ten confidence bins.

In [ ]:
coverages = np.linspace(1.0, 0.2, 81)
fig, axes = plt.subplots(
    1, len(FINAL_MODELS), figsize=(5 * len(FINAL_MODELS), 3.8), sharey=True
)
for ax, name in zip(axes, FINAL_MODELS, strict=True):
    for variant in VARIANTS:
        curve = np.mean(
            [
                risk_coverage(rob_probs[(name, seed, variant)], y_rob, coverages)
                for seed in SEEDS
            ],
            axis=0,
        )
        ax.plot(
            coverages,
            curve,
            label=variant,
            linestyle="--" if variant == "original" else "-",
        )
    ax.set_title(name)
    ax.set_xlabel("share of photos answered")
    ax.invert_xaxis()
    ax.grid(alpha=0.3)
axes[0].set_ylabel("accuracy of the answered photos")
axes[-1].legend(loc="lower left", fontsize=8)
finish("risk_coverage.png")

In [ ]:
rows = []
for name in FINAL_MODELS:
    for variant in VARIANTS:
        seed_stats = []
        for seed in SEEDS:
            probs = rob_probs[(name, seed, variant)]
            wrong = probs.argmax(1) != y_rob
            least_confident = np.argsort(probs.max(1), kind="stable")[
                : int(0.05 * len(y_rob))
            ]
            seed_stats.append(
                {
                    "errors": wrong.sum(),
                    "accuracy (all photos answered)": 1 - wrong.mean(),
                    "accuracy (95% answered)": risk_coverage(
                        probs, y_rob, np.array([0.95])
                    )[0],
                    "share of errors among the 5% least confident": (
                        wrong[least_confident].sum() / wrong.sum()
                        if wrong.any()
                        else np.nan
                    ),
                    "ECE": expected_calibration_error(probs, y_rob),
                }
            )
        rows.append(
            {
                "model": name,
                "variant": variant,
                **pd.DataFrame(seed_stats).mean().to_dict(),
            }
        )
confidence_table = pd.DataFrame(rows).set_index(["model", "variant"])
confidence_table.to_csv(out_path("confidence_abstention.csv"))
confidence_table.round(3)